## Build Camera-Site Covariates

Create a camera-level dataset with one row per Snapshot USA camera.
This notebook prepares camera-level covariates derived from Snapshot USA metadata, including:

- Total survey effort
- Habitat
- Development level
- Camera location (latitude and longitude)

Survey effort is aggregated across deployments while avoiding double-counting overlapping deployment periods


In [71]:
from pathlib import Path

import numpy as np
import pandas as pd

In [72]:
# Define input and output paths
# --------------------------------------------------

CLEANED = "../cleaned"
OUTPUT = "../../outputs/species_level_analysis"

# Input files
SSUSA_FILE = f"{CLEANED}/ssusa_cleaned.csv"
CAMERA_SPECIES_RF_BASE_FILE = f"{OUTPUT}/camera_species_rf_base.csv"

# Output file
CAMERA_COVARIATES_FILE = f"{OUTPUT}/camera_covariates.csv"

# Create the output directory if it does not exist
Path(OUTPUT).mkdir(parents=True, exist_ok=True)

In [73]:
# Load input datasets
# --------------------------------------------------

ssusa = pd.read_csv(
    SSUSA_FILE,
    low_memory=False,
)

camera_species_rf_base = pd.read_csv(
    CAMERA_SPECIES_RF_BASE_FILE
)

# Check dataset dimensions
print("SSUSA shape:", ssusa.shape)
print("Camera-species RF base shape:", camera_species_rf_base.shape)


SSUSA shape: (713319, 29)
Camera-species RF base shape: (484440, 4)


### Create Camera Identifiers

Create a unique `camera_fp_id` for each Snapshot USA record using its longitude and latitude.

The identifier uses coordinates rounded to eight decimal places and provides a common key for matching Snapshot USA records with cameras in the species-level analysis.

In [75]:
# Create the Camera Footprint Identifier for SSUSA records 

def make_camera_id(lon, lat):
    """Create a unique camera identifier from longitude and latitude."""
    return f"{lon:.8f}_{lat:.8f}"

ssusa["camera_fp_id"] = ssusa.apply(
    lambda row: make_camera_id(
        row["Longitude"],
        row["Latitude"]
    ),
    axis=1
)

### Identify the Cameras Included in the Species-Level Analysis

Compare the camera identifiers in the camera–species dataset with those generated from Snapshot USA.

This verifies that every camera used in the species-level analysis can be matched back to its Snapshot USA records before constructing camera-level covariates.


In [76]:
# Verify camera identifier matching
# --------------------------------------------------

# Unique camera IDs in Snapshot USA
ssusa_camera_ids = set(
    ssusa["camera_fp_id"].dropna().unique()
)

# Unique camera IDs used in the species-level analysis
rf_camera_ids = set(
    camera_species_rf_base["camera_fp_id"]
    .dropna()
    .unique()
)

# Identify matched and unmatched cameras
matched_camera_ids = rf_camera_ids.intersection(ssusa_camera_ids)
missing_camera_ids = rf_camera_ids.difference(ssusa_camera_ids)

# Verify matching
print(f"RF camera IDs: {len(rf_camera_ids):,}")
print(f"Matched camera IDs: {len(matched_camera_ids):,}")
print(f"Missing camera IDs: {len(missing_camera_ids):,}")

RF camera IDs: 7,340
Matched camera IDs: 7,340
Missing camera IDs: 0


In [77]:
# base camera-level dataset
# --------------------------------------------------

camera_covariates = (
    camera_species_rf_base[["camera_fp_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Verify number of unique cameras
print(f"Number of cameras: {len(camera_covariates):,}")



Number of cameras: 7,340


### Check Consistency of Categorical Site Variables

Some cameras may have multiple recorded values for `Habitat`, `Development_Level`, or `Feature_Type` across Snapshot USA records.

Count how many cameras have more than one non-missing value for each categorical variable before aggregating them to a single camera-level value.

In [78]:
# Select site and survey variables
# --------------------------------------------------

camera_site_data = ssusa[
    [
        "camera_fp_id",
        "Survey_Nights",
        "Habitat",
        "Development_Level",
        "Feature_Type",
        "Latitude",
        "Longitude",
        "Year",
    ]
]

In [79]:
# Check for multiple categorical values per camera
# --------------------------------------------------

categorical_variables = [
    "Habitat",
    "Development_Level",
    "Feature_Type",
]

for variable in categorical_variables:

    n_multiple = (
        camera_site_data
        .groupby("camera_fp_id")[variable]
        .nunique(dropna=True)
        .gt(1)
        .sum()
    )

    print(
        f"{variable}: "
        f"{n_multiple:,} cameras have multiple values"
    )

Habitat: 154 cameras have multiple values
Development_Level: 3 cameras have multiple values
Feature_Type: 68 cameras have multiple values


### Check Variation in Survey Effort

A camera may appear in multiple deployments with different numbers of survey nights.

Check how many cameras have more than one `Survey_Nights` value before calculating total survey effort at the camera level.

In [81]:
# Check variation in survey nights per camera
# --------------------------------------------------

survey_night_variation = (
    camera_site_data
    .groupby("camera_fp_id")["Survey_Nights"]
    .nunique()
)

print(
    "Cameras with multiple Survey_Nights values:",
    (survey_night_variation > 1).sum()
)

Cameras with multiple Survey_Nights values: 1264


In [83]:
# Prepare deployment periods
# --------------------------------------------------

# Convert deployment dates to datetime
ssusa["Start_Date"] = pd.to_datetime(ssusa["Start_Date"])
ssusa["End_Date"] = pd.to_datetime(ssusa["End_Date"])

# Keep one record for each unique deployment period
deployment_dates = (
    ssusa[
        [
            "camera_fp_id",
            "Deployment_ID",
            "Survey_Nights",
            "Start_Date",
            "End_Date",
        ]
    ]
    .drop_duplicates()
)

In [84]:
# Check for overlapping deployment periods
# --------------------------------------------------

def has_overlapping_deployments(group):
    """Return True if any deployment periods for a camera overlap."""

    # Sort deployments chronologically
    group = group.sort_values("Start_Date")

    previous_end = None

    for _, row in group.iterrows():

        # Check whether the current deployment starts
        # before the previous deployment has ended
        if previous_end is not None:
            if row["Start_Date"] < previous_end:
                return True

        previous_end = row["End_Date"]

    return False

In [86]:
# Summarize overlapping deployments by camera
# --------------------------------------------------

overlap_summary = (
    deployment_dates
    .groupby("camera_fp_id")
    .apply(
        has_overlapping_deployments,
        include_groups=False
    )
    .reset_index(name="has_overlap")
)

# Count cameras with and without overlapping deployments
print(
    overlap_summary["has_overlap"].value_counts()
)

has_overlap
False    7281
True       59
Name: count, dtype: int64


In [87]:
# Inspect cameras with overlapping deployments
# --------------------------------------------------

overlap_camera_ids = overlap_summary.loc[
    overlap_summary["has_overlap"],
    "camera_fp_id"
]

overlap_records = (
    deployment_dates[
        deployment_dates["camera_fp_id"].isin(overlap_camera_ids)
    ]
    .sort_values(
        ["camera_fp_id", "Start_Date"]
    )
)

overlap_records

,camera_fp_id,Deployment_ID,Survey_Nights,Start_Date,End_Date
204112,-102.69847200_33.40296500,TX_Grassland_YoakumDunes_21_dep_03 09/20/21,23,2021-09-20,2021-10-13
204183,-102.69847200_33.40296500,TX_Grassland_YoakumDunes_21_dep_03 09/20/2021,3,2021-09-20,2021-09-23
323308,-102.69847200_33.40296500,TX_Grassland_YoakumDunes_22_MM 09/28/2022,39,2022-09-28,2022-11-06
584656,-102.69847200_33.40296500,TX_Grassland_YoakumDunes_23_Middle 09/13/2023,48,2023-09-13,2023-10-31
665401,-103.24683700_34.25005900,NM_Grassland_Rural_ENMU_loc2c,56,2023-09-13,2023-11-08
...,...,...,...,...,...
213601,-99.87773300_32.23393900,Abilene State Park 12,61,2021-09-04,2021-11-04
280441,-99.88121100_32.23506900,Abilene State Park Texas 7 09/01/2021,57,2021-09-02,2021-10-29
281969,-99.88121100_32.23506900,Abilene State Park 7,57,2021-09-02,2021-10-29
323541,-99.88816900_32.23398600,TX_Grassland_Abilene_S. P. 11,15,2022-08-28,2022-09-12


In [88]:
# Identify overlapping deployment pairs
# --------------------------------------------------

def get_overlapping_pairs(group):
    """Return all overlapping deployment pairs for a camera."""

    # Sort deployments chronologically
    group = (
        group
        .sort_values(["Start_Date", "End_Date"])
        .reset_index(drop=True)
    )

    pairs = []

    # Compare each deployment with all later deployments
    for i in range(len(group)):
        for j in range(i + 1, len(group)):

            # An overlap exists when the later deployment starts
            # before the earlier deployment ends
            if group.loc[j, "Start_Date"] < group.loc[i, "End_Date"]:

                pairs.append(
                    {
                        "camera_fp_id": group.loc[i, "camera_fp_id"],
                        "deployment_1": group.loc[i, "Deployment_ID"],
                        "start_1": group.loc[i, "Start_Date"],
                        "end_1": group.loc[i, "End_Date"],
                        "survey_nights_1": group.loc[i, "Survey_Nights"],
                        "deployment_2": group.loc[j, "Deployment_ID"],
                        "start_2": group.loc[j, "Start_Date"],
                        "end_2": group.loc[j, "End_Date"],
                        "survey_nights_2": group.loc[j, "Survey_Nights"],
                    }
                )

    return pairs

In [89]:
# Build table of overlapping deployment pairs
# --------------------------------------------------

actual_overlap_pairs = []

for _, group in deployment_dates.groupby("camera_fp_id"):
    actual_overlap_pairs.extend(
        get_overlapping_pairs(group)
    )

# Convert collected overlap records to a DataFrame
actual_overlap_pairs = pd.DataFrame(
    actual_overlap_pairs
)

# Report total number of overlapping pairs
print(
    "Number of overlapping pairs:",
    len(actual_overlap_pairs)
)

Number of overlapping pairs: 89


In [90]:
# Resolve overlapping survey effort
# --------------------------------------------------

def resolve_survey_effort(group):
    """
    Calculate total survey effort for one camera.

    Overlapping deployment periods are grouped together and represented
    by the maximum Survey_Nights value within that overlap group.

    Non-overlapping deployment periods are added separately.
    """

    # Sort deployments chronologically
    group = (
        group
        .sort_values("Start_Date")
        .reset_index(drop=True)
    )

    total_effort = 0

    current_end = None
    current_max = 0

    for _, row in group.iterrows():

        start = row["Start_Date"]
        end = row["End_Date"]
        nights = row["Survey_Nights"]

        # Start the first overlap group
        if current_end is None:
            current_end = end
            current_max = nights

        # Deployment overlaps the current interval group
        elif start < current_end:
            current_end = max(current_end, end)
            current_max = max(current_max, nights)

        # No overlap: add the previous group's effort
        # and begin a new interval group
        else:
            total_effort += current_max
            current_end = end
            current_max = nights

    # Add effort from the final interval group
    total_effort += current_max

    return total_effort

In [92]:
# --------------------------------------------------
# Calculate total survey effort per camera
# --------------------------------------------------

camera_effort = (
    deployment_dates
    .groupby("camera_fp_id")
    .apply(
        resolve_survey_effort,
        include_groups=False
    )
    .reset_index(name="Total_Survey_Nights")
)

# Preview camera-level survey effort
camera_effort.head()

,camera_fp_id,Total_Survey_Nights
0,-100.23311000_35.92725000,18
1,-100.24179000_35.92768000,20
2,-100.24180200_35.92767900,49
3,-100.24181000_35.92775000,36
4,-100.24212710_35.92756100,4


In [93]:
# Validate camera-level survey effort
# --------------------------------------------------

print(f"Rows in camera_effort: {len(camera_effort):,}")
print(
    f"Unique cameras: "
    f"{camera_effort['camera_fp_id'].nunique():,}"
)

Rows in camera_effort: 7,340
Unique cameras: 7,340


### Assign Representative Site Categories

A camera may have multiple recorded values for `Habitat`, `Development_Level`, or `Feature_Type` across deployments.

For each camera, assign the most frequently recorded non-missing value (mode) for each variable. If multiple values are tied for the mode, use the first value returned by pandas.

In [94]:
# Assign representative site categories per camera
# --------------------------------------------------

def first_mode(series):
    """Return the most frequent non-missing value."""
    
    modes = series.dropna().mode()

    # Return missing if no valid category exists
    if modes.empty:
        return pd.NA

    # If multiple modes exist, use the first
    return modes.iloc[0]


camera_categories = (
    camera_site_data
    .groupby("camera_fp_id")
    .agg(
        Habitat=("Habitat", first_mode),
        Development_Level=("Development_Level", first_mode),
        Feature_Type=("Feature_Type", first_mode),
    )
    .reset_index()
)

# Verify one row per camera
print(f"Camera category rows: {len(camera_categories):,}")
print(
    f"Unique cameras: "
    f"{camera_categories['camera_fp_id'].nunique():,}"
)

camera_categories.head()

Camera category rows: 7,340
Unique cameras: 7,340


,camera_fp_id,Habitat,Development_Level,Feature_Type
0,-100.23311000_35.92725000,Grassland,Wild,Water source
1,-100.24179000_35.92768000,Grassland,Wild,Water source
2,-100.24180200_35.92767900,Grassland,Wild,Water source
3,-100.24181000_35.92775000,Grassland,Wild,Water source
4,-100.24212710_35.92756100,Grassland,Wild,Trail game


### Merge Camera-Level Covariates

Combine the camera-level datasets to create a single covariate table containing:
- Geographic coordinates
- Total survey effort
- Representative habitat
- Representative development level
- Representative feature type


In [101]:
# Merge camera-level covariates
# Merge camera-level covariates
# --------------------------------------------------

# Get one latitude/longitude pair per camera
camera_locations = (
    camera_site_data[
        ["camera_fp_id", "Latitude", "Longitude"]
    ]
    .drop_duplicates(subset="camera_fp_id")
)

# Merge location, survey effort, and site categories
# Start with one location record per camera
camera_covariates = (
    camera_site_data[
        ["camera_fp_id", "Latitude", "Longitude"]
    ]
    .drop_duplicates("camera_fp_id")
)

# Add total survey effort
camera_covariates = camera_covariates.merge(
    camera_effort,
    on="camera_fp_id",
    how="left",
)

# Add representative site categories
camera_covariates = camera_covariates.merge(
    camera_categories,
    on="camera_fp_id",
    how="left",
)

camera_covariates.head()

,camera_fp_id,Latitude,Longitude,Total_Survey_Nights,Habitat,Development_Level,Feature_Type
0,-136.22250000_59.42643000,59.42643,-136.22250,64,Forest,Wild,Water source
1,-135.92880000_59.39905000,59.39905,-135.92880,67,Forest,Wild,Water source
2,-135.84097000_59.38020000,59.38020,-135.84097,134,Forest,Wild,Water source
3,-136.30461000_59.42641000,59.42641,-136.30461,68,Forest,Wild,<NA>
4,-136.15141000_59.41195000,59.41195,-136.15141,68,Forest,Wild,<NA>


In [102]:
print(camera_covariates.columns.tolist())

['camera_fp_id', 'Latitude', 'Longitude', 'Total_Survey_Nights', 'Habitat', 'Development_Level', 'Feature_Type']


### Examine Feature Type Missingness

In [103]:
feature_summary = (
    camera_covariates["Feature_Type"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("Feature_Type")
    .reset_index(name="Camera_Count")
)

feature_summary

,Feature_Type,Camera_Count
0,Missing,5164
1,Trail game,792
2,Trail hiking,380
3,Other,354
4,Road dirt,316
5,Water source,284
6,Road paved,18
7,"Road dirt, Trail hiking",8
8,Burrow,7
9,Culvert,6


In [104]:
# Remove Feature_Type
# --------------------------------------------------

camera_covariates = camera_covariates.drop(
    columns="Feature_Type"
)

print(
    "Camera covariate table shape:",
    camera_covariates.shape
)

camera_covariates.head()

Camera covariate table shape: (7340, 6)


,camera_fp_id,Latitude,Longitude,Total_Survey_Nights,Habitat,Development_Level
0,-136.22250000_59.42643000,59.42643,-136.22250,64,Forest,Wild
1,-135.92880000_59.39905000,59.39905,-135.92880,67,Forest,Wild
2,-135.84097000_59.38020000,59.38020,-135.84097,134,Forest,Wild
3,-136.30461000_59.42641000,59.42641,-136.30461,68,Forest,Wild
4,-136.15141000_59.41195000,59.41195,-136.15141,68,Forest,Wild


### Save Camera-Level Covariates

Save the final camera-level covariate table for use in the species-level Random Forest analysis.

In [105]:
camera_covariates.to_csv(
    CAMERA_COVARIATES_FILE,
    index=False,
)

print(f"Saved {len(camera_covariates):,} cameras to:")
print(CAMERA_COVARIATES_FILE)

Saved 7,340 cameras to:
../../outputs/species_level_analysis/camera_covariates.csv
